# Notebook 4: Submission Validation
## IEOR E4010 — AI for Operations Research and Financial Engineering
### Columbia University, Spring 2026

---

## Purpose

This notebook validates your submission before you turn it in. It:

1. Loads your ML model (`submission/ml_model.pkl`) and runs predictions
2. Loads your DL model (`submission/lstm_model.pth`) and runs predictions
3. Loads your RL agent (`submission/rl_agent.zip`) and runs 10 evaluation episodes
4. Prints a pass/fail summary table
5. Creates `submission/student_info.json` — fill in your name and UNI

**Run all cells in order. Fix any errors before submitting.**

### Submission checklist
- [ ] `submission/ml_model.pkl` exists
- [ ] `submission/lstm_model.pth` exists
- [ ] `submission/rl_agent.zip` exists
- [ ] `submission/rl_config.json` exists
- [ ] `submission/student_info.json` filled in with your name and UNI
- [ ] All cells run without errors
- [ ] All metrics pass the minimum thresholds

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import subprocess, sys, os

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "xgboost>=2.0", "torch>=2.0", "gymnasium>=0.29",
    "stable-baselines3>=2.0", "scipy>=1.11",
    "pandas>=2.0", "numpy>=1.24", "scikit-learn>=1.3"])

import numpy as np
import pandas as pd
import torch
import json
import pickle
from sklearn.metrics import mean_absolute_error

os.makedirs("submission", exist_ok=True)
print("Setup complete.")

In [ ]:
# ── Data loading ───────────────────────────────────────────────────────────────
import shutil

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

for filename in ["nyiso_prices_weather_nyc_2025.csv",
                  "nyiso_rt_lbmp_nyc_2025_5min.csv",
                  "nyc_fleet_profiles.json"]:
    local_path = os.path.join(DATA_DIR, filename)
    if not os.path.exists(local_path):
        for parent in ["", "..", "../.."]:  
            candidate = os.path.join(parent, filename) if parent else filename
            if os.path.exists(candidate):
                shutil.copy(candidate, local_path)
                break

df_hourly = pd.read_csv(
    "data/nyiso_prices_weather_nyc_2025.csv", parse_dates=["timestamp"]
).sort_values("timestamp").reset_index(drop=True)

price_df_5min = pd.read_csv(
    "data/nyiso_rt_lbmp_nyc_2025_5min.csv", parse_dates=["timestamp"]
).sort_values("timestamp").reset_index(drop=True)

with open("data/nyc_fleet_profiles.json") as f:
    fleet_profiles = json.load(f)

# Validation split: months 9-12
split_date = pd.Timestamp("2025-09-01")
df_val = df_hourly[df_hourly["timestamp"] >= split_date].copy().reset_index(drop=True)

print(f"Hourly data: {len(df_hourly)} rows")
print(f"5-min data:  {len(price_df_5min)} rows")
print(f"Val set:     {len(df_val)} rows ({df_val['timestamp'].min().date()} to {df_val['timestamp'].max().date()})")

---
## Step 1: Fill in Student Information

In [ ]:
# ── FILL IN YOUR INFORMATION ─────────────────────────────────────────────────
student_info = {
    "name": "Your Name",         # ← CHANGE THIS
    "uni":  "abc1234",            # ← CHANGE THIS (Columbia UNI)
    "email": "abc1234@columbia.edu",  # ← CHANGE THIS
    "course": "IEOR E4010",
    "semester": "Spring 2026",
    "notes": "",  # Optional: describe your approach
}

# Save
with open("submission/student_info.json", "w") as f:
    json.dump(student_info, f, indent=2)

print("Student info saved:")
print(json.dumps(student_info, indent=2))

# Validation
if student_info["name"] == "Your Name" or student_info["uni"] == "abc1234":
    print("\nWARNING: Please update name and UNI in the cell above!")
else:
    print(f"\nHello, {student_info['name']} ({student_info['uni']})!")

---
## Step 2: Validate ML Model

In [ ]:
ml_status = "FAIL"
ml_mae    = np.nan

ML_MAE_THRESHOLD = 10.0  # $/MWh — must beat this to pass

try:
    with open("submission/ml_model.pkl", "rb") as f:
        ml_obj = pickle.load(f)
    
    # Verify required keys
    required_keys = ["model", "feature_cols", "target"]
    for k in required_keys:
        assert k in ml_obj, f"Missing key: {k}"
    
    feature_fn   = ml_obj.get("feature_fn", None)
    feature_cols = ml_obj["feature_cols"]
    
    # Apply features
    df_hourly["is_weekend"] = (df_hourly["timestamp"].dt.dayofweek >= 5).astype(int)
    df_hourly["hour"]       = df_hourly["timestamp"].dt.hour
    
    if feature_fn is not None:
        df_feats = feature_fn(df_hourly)
    else:
        df_feats = df_hourly.copy()
    
    val_mask = df_feats["timestamp"] >= split_date
    df_val_ml = df_feats[val_mask].dropna(subset=feature_cols)
    
    X_val_ml = df_val_ml[feature_cols].values
    y_val_ml = df_val_ml["rt_lbmp_mwh"].values
    
    preds_ml = ml_obj["model"].predict(X_val_ml)
    ml_mae   = mean_absolute_error(y_val_ml, preds_ml)
    
    if ml_mae < ML_MAE_THRESHOLD:
        ml_status = "PASS"
    
    print(f"ML model type:  {type(ml_obj['model']).__name__}")
    print(f"Feature count:  {len(feature_cols)}")
    print(f"Val samples:    {len(y_val_ml)}")
    print(f"MAE:            {ml_mae:.3f} $/MWh  (threshold: <{ML_MAE_THRESHOLD})")
    print(f"Status:         {ml_status}")

except FileNotFoundError:
    print("ERROR: submission/ml_model.pkl not found!")
    print("Run Notebook 1 and execute the model export cell.")
except Exception as e:
    print(f"ERROR loading ML model: {e}")

---
## Step 3: Validate DL Model

In [ ]:
dl_status = "FAIL"
dl_mae    = np.nan

DL_MAE_THRESHOLD = 12.0  # $/MWh

try:
    dl_obj = torch.load("submission/lstm_model.pth", map_location="cpu", weights_only=False)
    
    # Verify required keys
    for k in ["model_state_dict", "scaler_mean", "scaler_std", "seq_features", "seq_len"]:
        assert k in dl_obj, f"Missing key: {k}"
    
    seq_features  = dl_obj["seq_features"]
    seq_len       = dl_obj["seq_len"]
    scaler_mean   = pd.Series(dl_obj["scaler_mean"])
    scaler_std    = pd.Series(dl_obj["scaler_std"])
    target_mean   = dl_obj.get("target_mean", 0.0)
    target_std    = dl_obj.get("target_std",  1.0)
    model_config  = dl_obj.get("model_config", {})
    model_class   = dl_obj.get("model_class", "ImprovedForecaster")
    
    # Reconstruct model — try to import class from notebook context
    # Since we may not have the class defined here, use a generic rebuild
    import torch.nn as nn
    
    class _GRUAttention(nn.Module):
        """Matches ImprovedForecaster from Notebook 2."""
        def __init__(self, input_size, hidden_size, num_layers=2, dropout=0.15):
            super().__init__()
            self.hidden_size = hidden_size
            self.gru = nn.GRU(input_size, hidden_size, num_layers,
                              batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
            self.attention_fc = nn.Linear(hidden_size, 1, bias=False)
            self.fc1     = nn.Linear(hidden_size, 64)
            self.relu    = nn.ReLU()
            self.dropout = nn.Dropout(dropout)
            self.fc2     = nn.Linear(64, 1)
        def forward(self, x):
            out, _ = self.gru(x)
            scores  = self.attention_fc(out) / (self.hidden_size ** 0.5)
            weights = torch.softmax(scores, dim=1)
            context = (weights * out).sum(dim=1)
            h = self.relu(self.fc1(self.dropout(context)))
            return self.fc2(h).squeeze(-1)
    
    class _LSTMModel(nn.Module):
        """Matches LSTMForecaster from Notebook 2."""
        def __init__(self, input_size, hidden_size, num_layers=2, dropout=0.1):
            super().__init__()
            self.hidden_size = hidden_size
            self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                                batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
            self.dropout = nn.Dropout(dropout)
            self.fc = nn.Linear(hidden_size, 1)
        def forward(self, x):
            out, _ = self.lstm(x)
            return self.fc(self.dropout(out[:, -1, :])).squeeze(-1)
    
    in_size  = model_config.get("input_size", len(seq_features))
    hid_size = model_config.get("hidden_size", 128)
    n_layers = model_config.get("num_layers", 2)
    dropout  = model_config.get("dropout", 0.15)
    
    if "LSTM" in model_class:
        dl_model = _LSTMModel(in_size, hid_size, n_layers, dropout)
    else:
        dl_model = _GRUAttention(in_size, hid_size, n_layers, dropout)
    
    dl_model.load_state_dict(dl_obj["model_state_dict"])
    dl_model.eval()
    
    # Prepare validation sequences
    df_hourly_dl = df_hourly.copy()
    df_hourly_dl["is_weekend"] = (df_hourly_dl["timestamp"].dt.dayofweek >= 5).astype(float)
    
    val_normed = (df_hourly_dl[seq_features] - scaler_mean) / scaler_std
    target_normed = (df_hourly_dl["rt_lbmp_mwh"] - target_mean) / target_std
    
    # Build sequences for validation period
    val_start_idx = df_hourly_dl[df_hourly_dl["timestamp"] >= split_date].index[0]
    start = max(0, val_start_idx - seq_len)
    
    feat_arr   = val_normed.values[start:].astype(np.float32)
    target_arr = target_normed.values[start:].astype(np.float32)
    
    X_seq, y_seq = [], []
    for i in range(len(feat_arr) - seq_len):
        X_seq.append(feat_arr[i : i + seq_len])
        y_seq.append(target_arr[i + seq_len])
    
    X_seq = np.stack(X_seq)
    y_seq = np.array(y_seq)
    
    # Only evaluate on the actual validation period
    n_pre = val_start_idx - start
    X_val_dl = X_seq[n_pre:]
    y_val_dl = y_seq[n_pre:]
    y_val_raw_dl = df_hourly_dl["rt_lbmp_mwh"].values[val_start_idx + seq_len - n_pre : val_start_idx + seq_len - n_pre + len(y_val_dl)]
    
    with torch.no_grad():
        preds_norm = dl_model(torch.from_numpy(X_val_dl)).numpy()
    preds_dl  = preds_norm * target_std + target_mean
    
    # Use actual raw targets from the dataset
    y_true_dl = df_hourly_dl["rt_lbmp_mwh"].values[
        val_start_idx : val_start_idx + len(preds_dl)
    ]
    dl_mae = mean_absolute_error(y_true_dl[:len(preds_dl)], preds_dl[:len(y_true_dl)])
    
    if dl_mae < DL_MAE_THRESHOLD:
        dl_status = "PASS"
    
    print(f"DL model class: {model_class}")
    print(f"Seq length:     {seq_len}")
    print(f"Features:       {seq_features}")
    print(f"Val samples:    {len(preds_dl)}")
    print(f"MAE:            {dl_mae:.3f} $/MWh  (threshold: <{DL_MAE_THRESHOLD})")
    print(f"Status:         {dl_status}")

except FileNotFoundError:
    print("ERROR: submission/lstm_model.pth not found!")
    print("Run Notebook 2 and execute the model export cell.")
except Exception as e:
    import traceback
    print(f"ERROR loading DL model: {e}")
    traceback.print_exc()

---
## Step 4: Validate RL Agent

In [ ]:
# Re-define EVChargingEnv (same as Notebook 3 — needed to load the agent)
import gymnasium as gym
from scipy.stats import truncnorm


class EVChargingEnv(gym.Env):
    metadata = {"render_modes": []}
    VEHICLE_BATTERY_KWH = 76.0; FLEET_SIZE = 50; N_FAST = 10; N_SLOW = 10
    MAX_FAST_KW = 150.0; MAX_SLOW_KW = 11.0; BESS_CAPACITY = 1000.0
    BESS_MAX_DISCHARGE_KW = 1500.0; BESS_MAX_CHARGE_KW = 500.0
    BESS_EFF = 0.92; DC_EFF = 0.90; SLOW_EFF = 0.93; GRID_MAX_KW = 500.0
    TARGET_SOC = 0.70; DT = 5/60; N_STEPS = 288; DEADLINE_PENALTY = 50.0
    DEMAND_CHARGE_SUMMER = 53.60; DEMAND_CHARGE_ALL = 41.24

    def __init__(self, price_df_5min, fleet_profiles, seed=None,
                 reward_weights=None, forecast_fn=None, n_forecast_steps=0):
        super().__init__()
        self.price_df = price_df_5min.reset_index(drop=True)
        self.fleet_profiles = fleet_profiles
        self.forecast_fn = forecast_fn
        self.n_forecast_steps = n_forecast_steps
        self.rw = reward_weights or {"electricity": 1.0, "opportunity": 1.0,
                                      "deadline": 1.0, "peak_demand": 1.0}
        n_obs = 1 + (self.N_FAST*3) + (self.N_SLOW*3) + 1 + 1 + 1 + 1 + n_forecast_steps
        self.observation_space = gym.spaces.Box(low=-1.0, high=1.0, shape=(n_obs,), dtype=np.float32)
        self.action_space = gym.spaces.Box(low=0.0, high=1.0, shape=(21,), dtype=np.float32)
        self.np_random = np.random.default_rng(seed)
        self._episode_days = self._get_unique_days()

    def _get_unique_days(self):
        self.price_df["_date"] = pd.to_datetime(self.price_df["timestamp"]).dt.date
        return self.price_df["_date"].unique().tolist()

    def reset(self, seed=None, options=None):
        if seed is not None: self.np_random = np.random.default_rng(seed)
        day_idx = int(self.np_random.integers(0, len(self._episode_days)))
        self._episode_day = self._episode_days[day_idx]
        day_mask = self.price_df["_date"] == self._episode_day
        day_data = self.price_df[day_mask].reset_index(drop=True)
        if len(day_data) < self.N_STEPS:
            pad = self.N_STEPS - len(day_data)
            day_data = pd.concat([day_data]+[day_data.iloc[[-1]]]*pad, ignore_index=True)
        self._prices_kwh = day_data["rt_lbmp_kwh"].values[:self.N_STEPS].astype(np.float32)
        ts0 = pd.Timestamp(self._episode_day)
        self._day_of_week = ts0.dayofweek; self._is_weekend = ts0.dayofweek >= 5
        self._month = ts0.month
        self._bess_soc = float(self.np_random.uniform(0.30, 0.90))
        self._fast_slots = [None]*self.N_FAST; self._slow_slots = [None]*self.N_SLOW
        self._vehicle_queue = []
        for _ in range(int(self.np_random.integers(3, 12))):
            v = self._new_vehicle(0)
            placed = False
            for i in range(self.N_SLOW):
                if self._slow_slots[i] is None: self._slow_slots[i]=v; placed=True; break
            if not placed:
                for i in range(self.N_FAST):
                    if self._fast_slots[i] is None: self._fast_slots[i]=v; placed=True; break
            if not placed: self._vehicle_queue.append(v)
        self._step_idx=0; self._max_grid_draw=0.0; self._total_cost=0.0
        self._total_opp_cost=0.0; self._missed_deadlines=0; self._prev_grid_draw=0.0
        return self._get_obs(), {}

    def _new_vehicle(self, step):
        sp = self.fleet_profiles["arrival_soc"]
        a=(sp["min"]-sp["mean"])/sp["std"]; b=(sp["max"]-sp["mean"])/sp["std"]
        soc=float(truncnorm.rvs(a,b,loc=sp["mean"],scale=sp["std"],
                                 random_state=int(self.np_random.integers(0,2**30))))
        soc=float(np.clip(soc,sp["min"],sp["max"]))
        hour=(step//12)%24; dk="weekend" if self._is_weekend else "weekday"
        demand=self.fleet_profiles["ride_demand"][dk][hour]
        mxd=max(self.fleet_profiles["ride_demand"]["weekday"])
        df=demand/mxd
        dep=int(self.np_random.integers(4,18)) if df>0.6 else int(self.np_random.integers(24,96))
        return {"soc":soc,"depart_step":min(step+dep,self.N_STEPS-1),"arrival_step":step}

    def _process_arrivals(self):
        hour=(self._step_idx//12)%24; dk="weekend" if self._is_weekend else "weekday"
        lam=self.fleet_profiles["depot_arrival_rate"][dk][hour]/12
        for _ in range(int(self.np_random.poisson(lam))):
            self._vehicle_queue.append(self._new_vehicle(self._step_idx))

    def _assign_from_queue(self):
        for i in range(self.N_SLOW):
            if self._slow_slots[i] is None and self._vehicle_queue:
                self._slow_slots[i]=self._vehicle_queue.pop(0)
        for i in range(self.N_FAST):
            if self._fast_slots[i] is None and self._vehicle_queue:
                self._fast_slots[i]=self._vehicle_queue.pop(0)

    def _get_obs(self):
        obs=[float(self._bess_soc)*2-1]
        for slot in self._fast_slots:
            if slot is None: obs.extend([0.,0.,0.])
            else:
                sl=max(0,slot["depart_step"]-self._step_idx)
                obs.extend([1.,float(slot["soc"])*2-1,float(min(sl/96,1.))*2-1])
        for slot in self._slow_slots:
            if slot is None: obs.extend([0.,0.,0.])
            else:
                sl=max(0,slot["depart_step"]-self._step_idx)
                obs.extend([1.,float(slot["soc"])*2-1,float(min(sl/96,1.))*2-1])
        price=self._prices_kwh[self._step_idx] if self._step_idx<self.N_STEPS else 0.
        obs.extend([float(min(len(self._vehicle_queue)/10,1.))*2-1,
                    float(np.clip(price/0.20,-1.,1.)),
                    float(self._step_idx/self.N_STEPS)*2-1,
                    float(np.clip(self._prev_grid_draw/self.GRID_MAX_KW,0.,1.))*2-1])
        if self.n_forecast_steps>0:
            end=min(self._step_idx+self.n_forecast_steps+1,self.N_STEPS)
            fc=self._prices_kwh[self._step_idx+1:end]
            pad=self.n_forecast_steps-len(fc)
            fv=np.clip(np.pad(fc,(0,pad),mode="edge" if len(fc)>0 else "constant")/0.20,-1.,1.)
            obs.extend(fv.tolist())
        return np.array(obs,dtype=np.float32)

    def step(self, action):
        action=np.clip(action,0.,1.)
        fr=action[:self.N_FAST]; so=action[self.N_FAST:self.N_FAST+self.N_SLOW]; bcr=action[-1]
        pk=float(self._prices_kwh[self._step_idx])
        bck=float(bcr)*self.BESS_MAX_CHARGE_KW
        bck=min(bck,(1.-self._bess_soc)*self.BESS_CAPACITY/self.DT)
        bei=bck*self.DT; bes=bei*self.BESS_EFF
        fck=[0.]*self.N_FAST; bdk=0.; bav=self._bess_soc*self.BESS_CAPACITY/self.DT
        for i,slot in enumerate(self._fast_slots):
            if slot is None: continue
            rq=float(fr[i])*self.MAX_FAST_KW
            rm=max(0.,(self.TARGET_SOC*1.05-slot["soc"])*self.VEHICLE_BATTERY_KWH/self.DT)
            ak=max(0.,min(rq,rm,bav-bdk)); fck[i]=ak; bdk+=ak
        if bdk>self.BESS_MAX_DISCHARGE_KW:
            sc=self.BESS_MAX_DISCHARGE_KW/bdk; fck=[k*sc for k in fck]; bdk=self.BESS_MAX_DISCHARGE_KW
        sck=[0.]*self.N_SLOW; tsg=0.
        for i,slot in enumerate(self._slow_slots):
            if slot is None or float(so[i])<=0.5: continue
            rm=max(0.,(self.TARGET_SOC*1.05-slot["soc"])*self.VEHICLE_BATTERY_KWH/self.DT)
            ak=min(self.MAX_SLOW_KW,rm); sck[i]=ak; tsg+=ak
        gdk=bck+tsg
        if gdk>self.GRID_MAX_KW:
            ex=gdk-self.GRID_MAX_KW; bck=max(0.,bck-ex); bei=bck*self.DT; bes=bei*self.BESS_EFF; gdk=bck+tsg
        beo=bdk*self.DT
        self._bess_soc=float(np.clip(self._bess_soc+(bes-beo)/self.BESS_CAPACITY,0.,1.))
        for i,slot in enumerate(self._fast_slots):
            if slot is not None and fck[i]>0:
                self._fast_slots[i]["soc"]=min(1.,slot["soc"]+fck[i]*self.DC_EFF*self.DT/self.VEHICLE_BATTERY_KWH)
        for i,slot in enumerate(self._slow_slots):
            if slot is not None and sck[i]>0:
                self._slow_slots[i]["soc"]=min(1.,slot["soc"]+sck[i]*self.SLOW_EFF*self.DT/self.VEHICLE_BATTERY_KWH)
        self._max_grid_draw=max(self._max_grid_draw,gdk); self._prev_grid_draw=gdk
        ec=gdk*pk*self.DT; self._total_cost+=ec
        hour=(self._step_idx//12)%24; dk="weekend" if self._is_weekend else "weekday"
        ocp=self.fleet_profiles["opportunity_cost"][dk][hour]
        nic=sum(1 for s in self._fast_slots+self._slow_slots if s is not None and s["soc"]>=self.TARGET_SOC)
        oc=nic*ocp*self.DT; self._total_opp_cost+=oc
        dp=0.
        for slots in [self._fast_slots,self._slow_slots]:
            for i in range(len(slots)):
                if slots[i] is not None and slots[i]["depart_step"]<=self._step_idx:
                    v=slots[i]
                    if v["soc"]<self.TARGET_SOC: dp+=self.DEADLINE_PENALTY*(self.TARGET_SOC-v["soc"])*100; self._missed_deadlines+=1
                    slots[i]=None
        self._process_arrivals(); self._assign_from_queue()
        reward=(-self.rw["electricity"]*ec-self.rw["opportunity"]*oc-self.rw["deadline"]*dp)
        self._step_idx+=1; terminated=self._step_idx>=self.N_STEPS
        if terminated:
            rate=self.DEMAND_CHARGE_SUMMER if self._month in[6,7,8,9] else self.DEMAND_CHARGE_ALL
            dc=self._max_grid_draw*rate; reward-=self.rw["peak_demand"]*dc
        info={"elec_cost":ec,"opp_cost":oc,"deadline_penalty":dp,"bess_soc":self._bess_soc,
              "grid_draw_kw":gdk,"n_idle_charged":nic,"queue_length":len(self._vehicle_queue)}
        if terminated:
            info["total_cost"]=self._total_cost; info["missed_deadlines"]=self._missed_deadlines
            info["max_grid_draw_kw"]=self._max_grid_draw
            info["demand_charge"]=self._max_grid_draw*(self.DEMAND_CHARGE_SUMMER if self._month in[6,7,8,9] else self.DEMAND_CHARGE_ALL)
        obs=self._get_obs() if not terminated else np.zeros(self.observation_space.shape,dtype=np.float32)
        return obs,reward,terminated,False,info


class EVChargingEnvCustomReward(EVChargingEnv):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.rw = {"electricity":1.0,"opportunity":0.5,"deadline":3.0,"peak_demand":2.0}
        self._prev_prev_grid = 0.0
    def reset(self, seed=None, options=None):
        obs, info = super().reset(seed=seed, options=options)
        self._prev_prev_grid = 0.0
        return obs, info
    def step(self, action):
        obs,reward,terminated,truncated,info = super().step(action)
        if self._bess_soc>0.30: reward+=0.3*self.DT
        cd=info["grid_draw_kw"]; dc=abs(cd-self._prev_prev_grid)
        reward-=0.005*dc*self.DT; self._prev_prev_grid=getattr(self,"_prev_grid_draw",cd)
        return obs,reward,terminated,truncated,info


print("EVChargingEnv and subclass re-defined for this notebook.")

In [ ]:
from stable_baselines3 import PPO

rl_status   = "FAIL"
rl_mean_cost = np.nan
RL_COST_THRESHOLD = 5000.0  # $ per episode — must be below this to pass

try:
    # Load RL config to know which env class and forecast steps were used
    with open("submission/rl_config.json") as f:
        rl_config = json.load(f)
    
    n_fc = rl_config.get("n_forecast_steps", 0)
    print(f"RL config: forecast_variant={rl_config.get('forecast_variant','blind')}, "
          f"n_forecast_steps={n_fc}, "
          f"training_timesteps={rl_config.get('training_timesteps','?')}")
    
    # Create an eval env matching the training env
    eval_env = EVChargingEnvCustomReward(
        price_df_5min, fleet_profiles, seed=9999,
        n_forecast_steps=n_fc
    )
    
    # Load model
    model_rl = PPO.load("submission/rl_agent", env=eval_env)
    print("RL agent loaded successfully.")
    
    # Run 10 evaluation episodes
    rng = np.random.default_rng(9999)
    episode_costs = []
    missed_list   = []
    
    for ep in range(10):
        ep_seed = int(rng.integers(0, 10000))
        obs, _ = eval_env.reset(seed=ep_seed)
        done = False
        ep_info = {}
        for _ in range(288):
            action, _ = model_rl.predict(obs, deterministic=True)
            obs, r, done, _, info = eval_env.step(action)
            if done:
                ep_info = info
                break
        episode_costs.append(ep_info.get("total_cost", np.nan))
        missed_list.append(ep_info.get("missed_deadlines", np.nan))
    
    rl_mean_cost = float(np.nanmean(episode_costs))
    rl_mean_miss = float(np.nanmean(missed_list))
    
    if rl_mean_cost < RL_COST_THRESHOLD:
        rl_status = "PASS"
    
    print(f"\nRL evaluation (10 episodes):")
    print(f"  Mean cost:    ${rl_mean_cost:.2f}  (threshold: <${RL_COST_THRESHOLD:.0f})")
    print(f"  Mean missed:  {rl_mean_miss:.1f} departures per episode")
    print(f"  Status:       {rl_status}")

except FileNotFoundError as e:
    print(f"ERROR: {e}")
    print("Run Notebook 3 and execute the model export cell.")
except Exception as e:
    import traceback
    print(f"ERROR loading RL agent: {e}")
    traceback.print_exc()

---
## Step 5: Final Summary

In [ ]:
import textwrap

print("=" * 60)
print("SUBMISSION VALIDATION SUMMARY")
print(f"Student: {student_info['name']} ({student_info['uni']})")
print("=" * 60)
print()
print(f"{'Component':<25} {'Metric':<20} {'Value':>10} {'Status':>8}")
print("-" * 65)

rows = [
    ("ML Forecasting",    f"MAE ($/MWh)",           f"{ml_mae:.3f}" if not np.isnan(ml_mae) else "N/A",    ml_status),
    ("DL Forecasting",    f"MAE ($/MWh)",           f"{dl_mae:.3f}" if not np.isnan(dl_mae) else "N/A",    dl_status),
    ("RL Charging",       f"Mean cost ($)",          f"{rl_mean_cost:.2f}" if not np.isnan(rl_mean_cost) else "N/A",  rl_status),
]

all_pass = True
for comp, metric, value, status in rows:
    emoji = "OK" if status == "PASS" else "XX"
    print(f"  {comp:<23} {metric:<20} {value:>10} [{emoji}] {status}")
    if status != "PASS":
        all_pass = False

print()
print("=" * 60)
if all_pass:
    print("ALL CHECKS PASSED — ready to submit!")
    print("\nFiles to submit (zip the submission/ directory):")
    print("  submission/ml_model.pkl")
    print("  submission/lstm_model.pth")
    print("  submission/rl_agent.zip")
    print("  submission/rl_config.json")
    print("  submission/student_info.json")
else:
    print("SOME CHECKS FAILED — review errors above.")
print("=" * 60)

In [ ]:
# ── List submission directory ─────────────────────────────────────────────────
print("\nContents of submission/ directory:")
for fname in sorted(os.listdir("submission")):
    fpath = os.path.join("submission", fname)
    size  = os.path.getsize(fpath)
    print(f"  {fname:<30} {size/1024:.1f} KB")